In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
print(df.shape)
print(df.columns)

(50000, 2)
Index(['review', 'sentiment'], dtype='object')


## NLP Preprocessing

In [3]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pooja\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Pooja\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
def preprocess(text):
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra spaces
    text = " ".join(text.split())
    
    # Tokenization
    tokens = text.split()
    
    # Stopwords removal
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

In [6]:
df['clean_text'] = df['review'].apply(preprocess)

In [7]:
df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


## Feature Engineering

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000)

X_bow = cv.fit_transform(df['clean_text']).toarray()

In [9]:
y = df['sentiment'].map({'positive':1, 'negative':0})

In [10]:
print(X_bow.shape)

(50000, 5000)


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

In [12]:
print(X_tfidf.shape)

(50000, 5000)


### Feature Engineering

- Converted text data into numerical form using Bag of Words (BoW)
- Applied TF-IDF vectorization for better feature representation

## Model Building

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2)

In [14]:
#MODEL 1: Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

In [15]:
#MODEL 2: Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

pred_nb = nb.predict(X_test)

In [16]:
#MODEL 3: Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)

## Model Evaluation

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, pred):
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall:", recall_score(y_test, pred))
    print("F1 Score:", f1_score(y_test, pred))
    print("-"*40)

In [20]:
print("Logistic Regression")
evaluate(y_test, pred_lr)
print("Naive Bayes")
evaluate(y_test, pred_nb)

print("Decision Tree")
evaluate(y_test, pred_dt)

Logistic Regression
Accuracy: 0.8817
Precision: 0.8700818075574601
Recall: 0.8964479229379891
F1 Score: 0.8830681031926461
----------------------------------------
Naive Bayes
Accuracy: 0.8481
Precision: 0.8405426661423515
Recall: 0.8579169175195666
F1 Score: 0.8491409275995631
----------------------------------------
Decision Tree
Accuracy: 0.7081
Precision: 0.7086534573392641
Recall: 0.7033915312061008
F1 Score: 0.7060126900997079
----------------------------------------


## Comparison & Insights

- Logistic Regression gave the best performance in terms of accuracy and F1 score.
- Naive Bayes was faster but slightly less accurate.
- Decision Tree showed lower performance and may overfit the data.
- TF-IDF performed better than Bag of Words for this task.

Conclusion:
Logistic Regression with TF-IDF is the best combination for this dataset.